[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/thermodynamics/semiconductor_devices/mosfet.ipynb)

# A MOSFET

This example computes carrier transport and electrostatic gating across the planar cross-section of a metal–oxide–semiconductor field-effect transistor (MOSFET). An acceptor-doped silicon substrate holds mobile holes beneath two donor-doped surface wells, the source and the drain, separated by a thin carrier-free oxide under a metal gate. Raising the gate voltage repels holes from the surface beneath the oxide and draws electrons in until, above threshold, an inversion channel of electrons connects source to drain. A small drain voltage drives a current proportional to that voltage along the channel, while a larger drain voltage weakens the gate's pull near the drain until the channel pinches off there and the current saturates.

Planar spacetime across the cross-section is weighed by a null point weight `w` and joined with null quantity directions for electrons and holes, where the term ***parcel*** here means an extensive state combining carrier count with charge along `time`, so the same algebraic expressions govern one-dimensional segments and two-dimensional triangles alike. Pairing the potential plane `potentials & electron` reads the electron quasi-Fermi level, wedging with the spatial gradient forms one combined drift-diffusion drive `-(gradient ^ potentials)`, and `drive.commutator(electron)` gives isotropic drift in the plane without summing over coordinate axes. Because the gate oxide carries electric displacement in Gauss's law while its carrier conductivity vanishes, capacitive coupling between the gate contact and the inversion channel emerges directly from the coupled conservation balance; see also [the 1D p–n junction](diode.ipynb) and [solar-cell finger spacing](solar_cell.ipynb).

In vector-calculus drift–diffusion notation on the $(x, y)$ cross-section, Poisson's equation $-\partial_x(\epsilon\partial_x\psi) - \partial_y(\epsilon\partial_y\psi) = q(p - n + N_D^+ - N_A^-)$ in the silicon couples across the oxide interface via continuity of $\epsilon\partial_y\psi$ to Laplace's equation $-\nabla^2\psi = 0$ in the insulator, while electron and hole current components $J_{n,x}, J_{n,y}, J_{p,x}, J_{p,y}$ are assembled coordinate by coordinate from $-q\mu_n n\nabla\psi + q D_n\nabla n$ and $-q\mu_p p\nabla\psi - q D_p\nabla p$.

In [ ]:
# In Colab, install numga and its examples from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ https://github.com/EelcoHoogendoorn/numga/archive/HEAD.tar.gz"

In [ ]:
%matplotlib inline
from collections.abc import Iterator

import numpy as np

from examples.thermodynamics.semiconductor_devices import render, scenarios
from examples.util.animation import show_states
from numga import Algebra, NumpyContext, stack
from numga.sparse import spdiag

# Planar projective spacetime weighed by w0, joined with null quantity directions for electrons and holes.
ga = Algebra("x+y+t-w0") * Algebra("n0p0")
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                                                   # Scalar+
Point = ga.gatype.antivector()                                                # Antivector-
Plane = ga.gatype.vector()                                                    # Vector+
Upright = Plane.derive.within(Algebra("x+y+"))                                # Vector+
Potentials = Plane.derive.within(Algebra("t-n0p0"))                           # Vector+
Drive = ga.gatype(Upright ^ Potentials)                                       # Bivector+
# Unit charge along time and charged electron and hole parcels.
time = -mv.t.dual()                                                           # [] Point-
electron = mv.n.dual() - time                                                 # [] Point-
hole = mv.p.dual() + time                                                     # [] Point-
pairs = electron + hole                                                       # [] Point-
carriers = stack((electron, hole))                                            # [2] Point-
counts = stack((mv.n.dual(), mv.p.dual()))                                    # [2] Point-
count_planes = stack((mv.n, mv.p))                                            # [2] Plane+
charges = carriers - counts                                                   # [2] Point-
# Span of time and carrier quantities along which purely spatial fields stay constant.
unread = time & counts[0] & counts[1]                                         # [] Span-

## Transistor cross-section

An acceptor-doped silicon substrate spans forty Debye lengths horizontally and twenty vertically beneath two surface donor wells twenty times denser, separated by a thin gate oxide under an overlapping metal gate. Lengths are scaled by the substrate Debye length, potentials by the thermal voltage, densities by the substrate acceptor density, and times by the Debye electron diffusion time. Metal source and drain plugs reach through the oxide onto the two donor wells while a body contact grounds the bottom boundary. Graded rows and mirrored columns form a triangulation refined immediately beneath the oxide interface and across both well junctions, partitioning the triangles between carrier-bearing silicon and carrier-free oxide.

In [ ]:
intrinsic = 1e-6
mobilities = np.array([1.0, 1 / 3])
silicon_permittivity, oxide_permittivity = 1.0, 1 / 3
lifetime = 2.0
rate_constant = 1 / lifetime

In [ ]:
topology, coordinates, oxide, doping, contacts = scenarios.section()
V, E, F = topology.n_elements
vertices = (mv(Upright, coordinates) + mv.w).dual().field()                   # Point-[V]
held_indices, free_indices = scenarios.contact_partition(contacts, V)
held_reading = topology.reading(held_indices, 0)                              # Scalar+[V_held <- V]
free_reading = topology.reading(free_indices, 0)                              # Scalar+[V_free <- V]
semiconductor = mv.scalar(np.where(oxide, 0.0, 1.0)[:, None]).field()         # Scalar+[F]
permittivities = mv.scalar(np.where(oxide, oxide_permittivity, silicon_permittivity)[:, None]).field()   # Scalar+[F]
dopants = (time * doping).field()                                             # Point-[V]
render.section_figure(topology, vertices, semiconductor, dopants, contacts, scenarios.full_densities);

Blue donor wells at the left and right corners sit in the red acceptor-doped substrate beneath the grey gate oxide, dark metal gate, side contact plugs, and bottom body contact. A magnified surface strip immediately under the oxide displays the silicon interface tint at visible pixel thickness so the sub-Debye inversion channel can be seen directly as it forms.

## Coupled drift, diffusion, and electrostatics

The term ***simplex gradient*** here means the spatial covector at each corner of a triangle whose pairing with a planar displacement gives the linear change of that corner's interpolation weight across the triangle. Joining the opposite edge's two vertices with `unread`, dividing by its pairing with the corner itself, and projecting onto `Upright` yields the three corner simplex gradients of every triangle. Assembling those covectors gives `gradient`, and pulling back `-(gradient ^ Potentials)` via `.adjugate()` satisfies `potentials & outflow(flux) == drive_map(potentials) & flux`, summing triangle fluxes into vertex dual-cell outflows.

Each vertex's dual area collects one third of every incident silicon triangle while excluding oxide triangles so the insulator holds zero mobile charge. Along `time`, Gauss's law couples electric displacement across both silicon and oxide triangles to the net dual-cell charge, transmitting gate voltage across the insulator to the silicon surface. Damped Newton iteration solves the steady-state dual-cell balance with each carrier equation normalized by local density so depleted regions remain well conditioned.

In [ ]:
corners = topology.incidence[2, 0] * vertices                                 # [3] Point-[F]
facing = topology.off[2, 0] * vertices                                        # [2, 3] Point-[F]
sides = facing[0] & facing[1]                                                 # [3] Line-[F]
# Joining the opposite edge with unread vanishes along that edge; normalizing at the corner and projecting onto
# Upright extracts each corner's planar simplex gradient.
gradients = ((sides & unread) / (sides & unread & corners)).cast(Upright)     # [3] Upright+[F]
areas = (corners[0] & corners[1] & corners[2] & unread).norm() / 2            # Scalar+[F]
corner_mean = topology.incidence[2, 0].mean(axis=0)                           # Scalar+[F <- V]
dual_areas = corner_mean.adjugate() * (areas * semiconductor)                 # Scalar+[V]
gradient = (spdiag(gradients) * topology.incidence[2, 0]).sum(axis=0)         # Upright+[F <- V]
drive_map = -(gradient ^ Potentials)                                          # (Drive+ <- Potentials+)[F <- V]
outflow = drive_map.adjugate()                                                # (Point- <- Flux-)[V <- F]

In [ ]:
unit_conductances = (semiconductor * mobilities) * Drive.commutator(carriers).regressive(carriers)   # [2] (Flux- <- Drive+)[F]
displacement = permittivities * Drive.commutator(time).regressive(time)       # (Flux- <- Drive+)[F]


def carrier_densities(potentials: Potentials) -> Scalar:
    """Boltzmann's law relates the electron and hole densities (`[2] Scalar+[V]`) to the chemical potentials."""
    return intrinsic * (potentials & counts).exp()


def steady_states(potentials: Potentials, contacts: Potentials) -> Iterator[Potentials]:
    """Yields the steady-state potential field (`Potentials+[V]`) for each frame of held `contacts`
    (`[n_frames] Potentials+[V_held]`) by damped Newton iteration."""
    for contact in contacts:
        potentials = free_reading.adjugate() * (free_reading * potentials) + held_reading.adjugate() * contact
        while True:
            densities = carrier_densities(potentials)                         # [2] Scalar+[V]
            drive = drive_map(potentials)                                     # Drive+[F]
            conductance = ((corner_mean * densities) * unit_conductances).sum(axis=0)   # (Flux- <- Drive+)[F]
            rates = rate_constant * (densities[0] * densities[1] - intrinsic ** 2)   # Scalar+[V]
            charge = (densities * charges).sum(axis=0) + dopants              # Point-[V]
            gauss = outflow(areas * displacement(drive)) - dual_areas * charge   # Point-[V]
            residual = outflow(areas * conductance(drive)) + dual_areas * rates * pairs + gauss   # Point-[V]
            capacitance = dual_areas * (densities * -charges * (Potentials & counts)).sum(axis=0)
            drive_response = (areas * (conductance + displacement))(drive_map)
            density_response = (spdiag(areas * unit_conductances(drive)) * corner_mean
                                * spdiag(densities * (Potentials & counts))).sum(axis=0)
            pair_response = dual_areas * rate_constant * densities[0] * densities[1] * (pairs * (Potentials & pairs))
            jacobian = capacitance + pair_response + outflow(drive_response + density_response)
            balance_weights = time * (mv.t & Point) + (counts * (count_planes & Point) / densities).sum(axis=0)
            free_jacobian = free_reading * balance_weights(jacobian) * free_reading.adjugate()
            step_change = -(free_reading.adjugate() * free_jacobian.solve(free_reading * balance_weights(residual)))
            chemical_shift = (counts & step_change).abs().sum(axis=0)         # Scalar+[V]
            potentials = potentials + step_change / (1 + chemical_shift / limit)
            if chemical_shift.max() < tolerance:
                break
        yield potentials

## Channel inversion and current saturation

Ohmic contacts hold the source, drain, and body at local charge neutrality with grounded source and body Fermi levels and a drain Fermi level lowered by the applied drain voltage, while the gate contact sets the electrostatic potential along the top of the oxide with substrate work function so bands start flat at zero bias. Holding a small drain bias while raising the gate voltage forms the inversion channel, after which the drain voltage sweeps across three gate levels (1.8 V, 2.2 V, and 2.6 V), reading drain current from the charge entering the source contact via `t & outflow(areas * flux)`. Running the cell below takes about ten seconds.

In [ ]:
limit, tolerance = 6.0, 0.1
frame_duration_ms = 40
gates, drains, curves = scenarios.gate_drain_schedule()
n_frames = len(gates)
volt = 1 / 0.02585
largest_drain, current_window = scenarios.largest_drain, scenarios.drain_current_window

In [ ]:
chemical = scenarios.neutral_potentials(doping)                               # [2, V]
substrate = scenarios.neutral_potentials(np.array(-1.0))[0]
source, drain, _, gate = contacts
on_gate = np.isin(np.arange(V), gate)                                         # [V]
neutral = (mv.t * np.where(on_gate, substrate, chemical[0]) + mv.n * np.where(on_gate, 0.0, chemical[0])
           + mv.p * np.where(on_gate, 0.0, chemical[1])).field()             # Potentials+[V]
contact_voltages = scenarios.transistor_voltages(contacts, gates, drains)     # [n_frames, V_held]
wired_contacts = held_reading * neutral + (mv.t * contact_voltages).field()   # [n_frames] Potentials+[V_held]


def terminal_readings(potentials: Potentials) -> tuple[Scalar, Scalar]:
    """Source-to-drain quasi-Fermi drop and source charge inflow determine the drain voltage and current (`[] Scalar+`
    each)."""
    levels = potentials & electron                                            # Scalar+[V]
    rise = (topology.reading(source, 0) * levels).sites.mean() - (topology.reading(drain, 0) * levels).sites.mean()
    conductance = ((corner_mean * carrier_densities(potentials)) * unit_conductances).sum(axis=0)   # (Flux- <- Drive+)[F]
    net_outflow = outflow(areas * conductance(drive_map(potentials)))         # Point-[V]
    return rise, -(topology.reading(source, 0) * (mv.t & net_outflow)).sites.sum()


switched = show_states(steady_states(neutral, wired_contacts), render.transistor_animation, topology, vertices,
                       semiconductor, contacts, carrier_densities, terminal_readings, scenarios.full_densities, curves,
                       largest_drain, current_window, volt, name="mosfet", duration_ms=frame_duration_ms)

As the gate voltage rises, majority holes retreat from the oxide interface and widen the white depletion zone beneath the gate until, past roughly 1.4 V, mobile electrons populate a blue inversion layer linking source and drain. Sweeping the drain voltage drives a linear ohmic current at low bias, widens the drain depletion zone until the inversion layer pinches off near the drain corner, and levels off each output curve at a saturation current that increases with gate overdrive, accompanied by slight channel-length modulation as the pinched-off region recedes toward the source.

In [ ]:
# checks
x_position = mv.x & vertices                                                  # Scalar+[V]


def sheet_density(potentials: Potentials, left: float, right: float) -> Scalar:
    """Dual-cell integration between `left` and `right` gives the sheet electron density (`[] Scalar+`)."""
    window = topology.reading(np.flatnonzero((x_position > left) & (x_position < right)), 0)
    return (window * (dual_areas * carrier_densities(potentials)[0])).sites.sum() / (right - left)


voltages, currents = (stack(readings) for readings in zip(*(terminal_readings(state) for state in switched)))
# Above threshold, mid-channel sheet electron density grows at the gate-oxide areal capacitance, while subthreshold
# current remains negligible.
n_rise_frames = scenarios.n_rise_frames
rise_indices = np.arange(n_rise_frames)
mid_sheet = stack([sheet_density(switched[frame], -4.0, 4.0) for frame in rise_indices])   # [n_rise_frames] Scalar+
gate_step = gates[rise_indices[-1]] - gates[rise_indices[-2]]
threshold = gates[rise_indices[-1]] - mid_sheet[-1] * gate_step / (mid_sheet[-1] - mid_sheet[-2])
strong = rise_indices[gates[rise_indices] > threshold + 6]
sheet_slope = ((mid_sheet[strong[1:]] - mid_sheet[strong[:-1]])
               / np.diff(gates[strong]) / (oxide_permittivity / scenarios.oxide_thickness))
assert len(strong) > 2 and bool(((sheet_slope - 1).abs() < 0.2).all())
below = rise_indices[gates[rise_indices] < threshold - 10]
assert len(below) > 0 and bool(currents[below].abs().max() < 1e-3 * currents[rise_indices[-1]])
# Each output curve is ohmic at small drain voltage and saturates at large drain voltage.
for curve in range(len(scenarios.gate_voltages)):
    curve_frames = np.flatnonzero(curves == curve)
    order = curve_frames[voltages[curve_frames].argsort()]
    conductance = currents[order[:2]] / voltages[order[:2]]
    assert (conductance[1] / conductance[0] - 1).abs() < 0.15
    tail_slope = (currents[order[-2:]] - currents[order[-3:-1]]) / (voltages[order[-2:]] - voltages[order[-3:-1]])
    assert (tail_slope < conductance[0] / 6).all()
# Along the first output curve, drain-side electron density matches source-side density in the linear regime and drops
# sharply once pinched off in saturation.
on_first = np.flatnonzero(curves == 0)
linear = on_first[int(voltages[on_first].argmin())]
saturated = on_first[int(voltages[on_first].argmax())]
assert sheet_density(switched[linear], 8.0, 11.0) > 0.8 * sheet_density(switched[linear], -11.0, -8.0)
assert sheet_density(switched[saturated], 8.0, 11.0) < 0.2 * sheet_density(switched[saturated], -11.0, -8.0)